# as-strided-noncontig-source — ex4: fix .view() with .contiguous()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five patterns around tensor memory layout that ramp from reading strides → recognizing transpose breaks contiguity → seeing `.view()` fail on non-contig → fixing it with `.contiguous()` → building a zero-copy sliding-window view via `as_strided`. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `as-strided-noncontig-source`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Strides and contiguity — quick refresher

**Stride** = number of elements to skip in storage to advance one step along that axis.
- A contiguous `(H, W)` tensor has stride `(W, 1)`.
- A contiguous `(B, C, H, W)` tensor has stride `(C*H*W, H*W, W, 1)`.

**Contiguity** = the strides match the row-major layout of the current shape.
- `.T` swaps strides but not data → the result is a view but not contiguous.
- `.view()` requires contiguous input — it never copies.
- `.reshape()` makes a view if possible, copies if not.
- `.contiguous()` forces a row-major copy if the tensor isn't already contiguous.

**`as_strided(size, stride)`** is the lowest-level view constructor — you provide the exact shape and stride pair. Bypasses all safety checks; trust the values you pass.

### Exercise 4 — fix .view() with .contiguous()

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `.contiguous()` before `.view()` to copy a non-contiguous view into a fresh row-major layout.
> Keywords: contiguous-copy, view-fix, row-major
> ```

**KCs targeted:** `contiguous-fixes-view`

Implement `ex4_flatten_transpose(x)` to flatten `x.T` to a 1-D tensor in row-major (C) order, the **transpose-then-flatten** layout.

Input shape: `(H, W)`. Output shape: `(H * W,)`. Order: read `x.T` row by row, top to bottom (so the result is `[x[0,0], x[1,0], x[2,0], ..., x[0,1], x[1,1], ...]` — column-major over the original).

**Strategy:** call `.contiguous()` on `x.T` first (materializes a fresh contiguous tensor with the transposed layout), then `.view(-1)`. Equivalent: `x.T.reshape(-1)` (PyTorch's reshape does the copy for you when needed). Both are accepted.

In [ ]:
def ex4_flatten_transpose(x: Tensor) -> Tensor:
    """Flatten x.T in row-major order. (H, W) → (H*W,)."""
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(12).reshape(3, 4).float()
    y = ex4_flatten_transpose(x)
    assert y.shape == (12,), f'expected (12,), got {tuple(y.shape)}'
    # Expected order: column-major over the original = [x[0,0],x[1,0],x[2,0], x[0,1],x[1,1],x[2,1], ...]
    expected = t.tensor([0., 4., 8., 1., 5., 9., 2., 6., 10., 3., 7., 11.])
    assert t.equal(y, expected), f'value mismatch: {y.tolist()} vs {expected.tolist()}'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_flatten_transpose(x: Tensor) -> Tensor:
    return x.T.contiguous().view(-1)
```

**`.contiguous()` semantics.** If the tensor is already contiguous, it's a no-op (returns `self`). Otherwise it allocates a fresh storage buffer and copies values into row-major order. After that any view operation is legal.

**Pythonic shortcut:** `x.T.reshape(-1)`. `.reshape()` tries to produce a view (zero copy) and falls back to copy if it can't. Most code uses `.reshape()` and forgets about `.contiguous()` entirely — knowing the distinction is mostly defensive.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()